# eSett Open Data API — BESS historical backfill

This notebook tests the four existing eSett dataset families in the BESS repo using eSett's public OpenAPI. Start with **one Swedish local day / one MBA**, inspect the rows, then explicitly opt into historical backfill. Data are retained as source JSON with UTC boundaries, not silently reclassified as measured site load, site PV, Nord Pool spot, or mFRR settlement data.

OpenAPI: https://api.opendata.esett.com/openapi?format=json. No token required. Queries use MBA **EIC codes**, not display names such as `SE3`. This notebook has not been run against the live API in the PR.

In [ ]:
%pip install requests pandas

In [ ]:
import json
from datetime import date, timedelta
from pathlib import Path
import pandas as pd
import sys

# Works when Jupyter is launched from repo root or from notebooks/
ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'esett_open_data.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'esett_open_data.py').exists(), 'Launch Jupyter from repository or notebooks/'
sys.path.insert(0, str(ROOT))

from scripts.esett_open_data import (ROUTES, backfill, fetch_day, local_day_window,
                                     mba_options, new_session, swedish_mba_codes, check_day, metering_grid_areas)
session = new_session()
SAMPLE_DAY = date(2026, 6, 24)
OUTPUT_ROOT = ROOT / 'data' / 'history' / 'source' / 'esett'

## 1. Discover Swedish MBA EIC codes

The four datasets can have different option lists. Inspect the returned names and codes before choosing a geographic scope. `/EXP18/MBAOptions` may have a nested JSON list.

In [ ]:
all_options = {}
swedish_codes = {}
for dataset in ROUTES:
    all_options[dataset] = mba_options(dataset, session)
    swedish_codes[dataset] = swedish_mba_codes(all_options[dataset])
    print(f'{dataset:18} Swedish EIC codes: {swedish_codes[dataset]}')
    if not swedish_codes[dataset]:
        print('  Could not automatically identify Sweden: inspect raw groups below')
        print(json.dumps(all_options[dataset][:2], indent=2)[:1500])

## 2. One-MBA / one-day smoke test

This deliberately selects **one** discovered Swedish MBA per dataset. It is a connectivity/schema smoke test, **not** a complete Swedish aggregation. A Swedish-local date may span UTC dates and contains 92, 96, or 100 intervals at DST transitions. API row counts may exceed the number of quarters for Load Profile because it is per MGA.

In [ ]:
ONE_MBA = {dataset: codes[:1] for dataset, codes in swedish_codes.items()}
assert all(ONE_MBA.values()), 'Inspect /MBAOptions and select EIC codes manually before fetching'
print('Local day UTC interval:', local_day_window(SAMPLE_DAY))

# Keep failures visible per dataset without blocking inspection of the others.
smoke_results = {}
for dataset, codes in ONE_MBA.items():
    try:
        result = backfill([dataset], {dataset: codes}, SAMPLE_DAY,
                          SAMPLE_DAY + timedelta(days=1), root=OUTPUT_ROOT,
                          session=session, pause_seconds=0.3)
        smoke_results[dataset] = result
    except Exception as exc:
        smoke_results[dataset] = str(exc)
        print(f'{dataset} failed: {type(exc).__name__}: {exc}')

## 2b. When load-profile API requests exceed provider limits

`/EXP18/LoadProfile` can return many meter-grid areas. If that smoke test fails with a row-limit error, discover MGA codes via `/EXP03/MeteringGridAreas` and query a single MGA. This is a **subset**, not the complete Sweden load-profile dataset.

In [ ]:
load_eic = ONE_MBA['load_profile'][0]
areas = metering_grid_areas(load_eic, session)
display(pd.DataFrame(areas).head(10))

# Uncomment after choosing a real code returned by the discovery request.
# chosen_mga = areas[0]['mgaCode']
# rows = fetch_day('load_profile', load_eic, SAMPLE_DAY, session, mga=chosen_mga)
# print(check_day(rows, SAMPLE_DAY))

## 3. Inspect records and coverage

Look for the fields documented in the OpenAPI schema: `timestampUTC`, `mba`, `solar`, `total`, `quantity`, `imblPurchasePrice`, etc. Preserve the original source payload and market-specific units. A 204/empty response should not be counted as fully covered.

In [ ]:
for dataset, codes in ONE_MBA.items():
    if isinstance(smoke_results.get(dataset), str):
        continue
    paths = sorted((OUTPUT_ROOT / dataset).rglob(f'{SAMPLE_DAY.isoformat()}.json'))
    print('\n', dataset, 'saved partitions:', len(paths))
    for path in paths[:1]:
        payload = json.loads(path.read_text(encoding='utf-8'))
        print('Audit:', payload['audit'])
        frame = pd.DataFrame(payload['records'])
        display(frame.head(5))
        print('API columns:', list(frame.columns))

## 4. Optional historical backfill (explicit opt-in)

Start with **one week**, then one month. Only enable two years after the one-day data schema, coverage, source restrictions and rate limits have been validated. Requests are daily and resumable: existing JSON partitions are skipped unless `overwrite=True`. For complete Swedish market context, select **all intended Sweden MBAs** per dataset and keep them distinct. For `load_profile`, the MGA-level output can be large; choose an MGA explicitly if necessary.

The window below matches the existing draft PR: 2024-07-01 through 2026-06-30 inclusive.

In [ ]:
RUN_BACKFILL = False  # Change only after validating the smoke test
FULL_SWEDEN = {dataset: codes for dataset, codes in swedish_codes.items()}
BACKFILL_START = date(2024, 7, 1)
BACKFILL_END_EXCLUSIVE = date(2026, 7, 1)

if RUN_BACKFILL:
    for dataset in ROUTES:
        results = backfill(
            [dataset], FULL_SWEDEN,
            BACKFILL_START, BACKFILL_END_EXCLUSIVE,
            root=OUTPUT_ROOT,
            session=session,
            pause_seconds=0.3,
        )
        print(dataset, pd.DataFrame(results).groupby('status').size().to_dict())

## Interpretation and integration

- `production` solar is a **market-wide solar production profile (MWh)**, not a metered 800-kW site PV trace. Summarize/normalize separately after verifying geographic coverage; do not combine MBA records incorrectly.
- `consumption` and `load_profile` describe market/area consumption, **not factory meter readings**. Do not write these directly as `site_load_kw`.
- `imbalance_prices` may inform regulation-price analysis but does **not** substitute for Nord Pool day-ahead spot prices, FCR-N/mFRR capacity auctions, or mFRR activation volumes.
- The historical ingestion framework in `bess_optimizer/data/history.py` expects normalized family-specific inputs. This notebook writes **source-preserving** partitions under `data/history/source/esett/`; no automatic mapping is performed until source semantics and API responses are verified.
- Avoid committing the large downloaded history. The repository ignores `data/history/source/esett/`.
